# Telco Customer Churn: Modelling Notebook

**Group Alpha:**
- Amira Aqila Afdhal
- Christian
- Ditya Ayu Anjani

# Import Libraries

In [30]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
# pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',500)
pd.set_option('display.max_colwidth', 200)
from scipy.stats import normaltest

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile

# Imbalance Dataset
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Modeling 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb

# Evaluation
from sklearn.model_selection import cross_validate,GridSearchCV,StratifiedKFold,train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import make_scorer,fbeta_score,recall_score, precision_score, balanced_accuracy_score
from sklearn.metrics import roc_auc_score, RocCurveDisplay

import pickle

## Data Preparation for Modelling

In [31]:
df = pd.read_csv('train_telco.csv')

In [32]:
df

,gender,AgeGroup,Partner,CLTV,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,PaymentMethod,tenure,MonthlyCharges,TotalCharges,Contract,Churn Reason,Churn
0,Male,Middle Age (30–59),True,4603,True,Yes,Fiber optic,Yes,Yes,Yes,Yes,No,No,False,Credit card (automatic),65,94.55,6078.75,Two year,-,False
1,Male,Under 30,False,2734,False,No phone service,DSL,No,No,Yes,Yes,No,No,False,Electronic check,26,35.75,1022.50,Month-to-month,-,False
2,Female,Under 30,True,4095,True,Yes,Fiber optic,No,Yes,Yes,Yes,No,No,False,Credit card (automatic),68,90.20,6297.65,Two year,-,False
3,Male,Middle Age (30–59),False,5828,True,No,Fiber optic,No,Yes,No,No,No,Yes,False,Electronic check,3,84.30,235.05,Month-to-month,-,False
4,Female,Middle Age (30–59),True,5565,False,No phone service,DSL,Yes,No,No,No,Yes,No,False,Bank transfer (automatic),49,40.65,2070.75,Month-to-month,-,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5620,Male,Senior (≥ 60),True,4871,True,No,Fiber optic,No,No,No,No,No,No,True,Bank transfer (automatic),65,70.95,4555.20,One year,-,False
5621,Female,Senior (≥ 60),False,4526,True,No,Fiber optic,No,Yes,No,No,No,No,True,Credit card (automatic),15,75.30,1147.45,Month-to-month,Competitor offered more data,True
5622,Female,Under 30,True,3570,True,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,True,Credit card (automatic),36,92.90,3379.25,Two year,-,False
5623,Female,Middle Age (30–59),True,5289,True,No,DSL,No,Yes,Yes,No,No,Yes,True,Mailed check,10,65.90,660.05,One year,-,False


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5625 entries, 0 to 5624
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            5625 non-null   object 
 1   AgeGroup          5625 non-null   object 
 2   Partner           5625 non-null   bool   
 3   CLTV              5625 non-null   int64  
 4   PhoneService      5625 non-null   bool   
 5   MultipleLines     5625 non-null   object 
 6   InternetService   5625 non-null   object 
 7   OnlineSecurity    5625 non-null   object 
 8   OnlineBackup      5625 non-null   object 
 9   DeviceProtection  5625 non-null   object 
 10  TechSupport       5625 non-null   object 
 11  StreamingTV       5625 non-null   object 
 12  StreamingMovies   5625 non-null   object 
 13  PaperlessBilling  5625 non-null   bool   
 14  PaymentMethod     5625 non-null   object 
 15  tenure            5625 non-null   int64  
 16  MonthlyCharges    5625 non-null   float64



### Feature Information

| Feature Name        | Description |
|---------------------|-------------|
| customerID          | Unique identifier for each customer |
| gender              | Customer's gender |
| AgeGroup            | Whether the customer is under 30 years old, middle age, or senior |
| Partner             | Whether the customer has a partner |
| PhoneService        | Whether the customer has phone service |
| MultipleLines       | Whether the customer has multiple phone lines |
| InternetService     | Type of internet service (DSL, Fiber optic, None) |
| OnlineSecurity      | Whether the customer has online security service |
| OnlineBackup        | Whether the customer has online backup service |
| DeviceProtection    | Whether the customer has device protection service |
| TechSupport         | Whether the customer has tech support service |
| StreamingTV         | Whether the customer has streaming TV service |
| StreamingMovies     | Whether the customer has streaming movie service |
| Contract            | Customer"s contract type (Month-to-month, One year, Two year) |
| PaperlessBilling    | Whether the customer is on paperless billing |
| PaymentMethod       | Customer"s payment method (e.g., Electronic check, Mailed check) |
| tenure              | Number of months the customer has stayed with the company |
| MonthlyCharges      | The amount charged to the customer monthly |
| TotalCharges        | The total amount charged to the customer |
| CLTV                | Customer Lifetime Value |
| Satisfaction Score  | Customer satisfaction rating |
| Churn Reason        | Reason the customer churned |
| Churn               | Whether the customer churned (Yes or No) |

For modelling our target variable is Churn

We can **drop**:
- `customerID` since this is only an identifier for each customer, this gives no information about the likelihood for the customer to churn.
- `Churn Reason` since this will lead to data leakage (only the customers who has churned has a churn reason)

### Categorical Feature Encoding

**1. Other features with only two values** -> Use `One Hot Encoding`. These are features which were mot transformed earlier for interpretability, but where encoding is now required for modelling: `gender`, `SeniorCitizen`, `Partner`,`PhoneService`,`PaperlessBilling`
\
\
**2. Features with 3-5 unique values** -> Use `One Hot Encoding`. These are categorical features with a small number of unique values: `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `Contract`, `PaymentMethod`

Features with One Hot Encoding:  'gender', 'AgeGroup', 'Partner', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod', 'Contract'

In [34]:
# Get columns with low unique values (potential categorical columns)
categorical_cols = [col for col in df.columns if (df[col].nunique() > 1 and df[col].nunique() < 3)]  # Adjust threshold as needed
print(categorical_cols)

['gender', 'Partner', 'PhoneService', 'PaperlessBilling', 'Churn']


In [35]:
df.select_dtypes(include='number').columns

Index(['CLTV', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

In [36]:
df['PaymentMethod'].nunique()

4

### Numerical Features

**To do:**
1. `tenure` → Right-skewed; apply **log transform or bin**, then **scale**
1. `MonthlyCharges` → Slightly bimodal and skewed; **scale** or **bin into pricing tiers**
1. `CLTV` → Moderately skewed and spread out; optionally apply **log transform**, then **scale**
1. `Satisfaction Score` → Integer values 1–5; leave it as is since this is sufficient for Tree-based models (e.g., Random Forest, XGBoost)

**Convert:**
  * `SeniorCitizen` → to **categorical**
  * `Number of Dependents`, `Satisfaction Score` → to **ordinal or categorical**

**Transform and scale:**
  * `tenure`, `Population`, `CLTV` → **log transform**, then **scale**
  * `MonthlyCharges` → **scale** (StandardScaler or RobustScaler)

**Correlated features:**
  * Keep features like `tenure`, `CLTV`, and `Satisfaction Score`
  * Explore **feature interactions** (e.g., `tenure` & `CLTV`, `SeniorCitizen` & `MonthlyCharges`)
  * Monitor **multicollinearity** in correlated pairs

**Handle categorical cardinality:**
  * Drop `customerID` (identifier)
  * `City` → **group or frequency encode**
  * `Churn Reason`, `PaymentMethod`, `OnlineBackup` → **one-hot encode**

**Data quality:**
  * **Impute or drop** columns with missing values
  * Handle outliers using **clipping**, **log transformation**, or **robust scaling**

In [37]:
x = df.drop(columns=['Churn', 'Churn Reason'])
y = df['Churn']

In [38]:
x_train,x_test,y_train,y_test=train_test_split(x,y,stratify=y,test_size=0.2,random_state=42)

In [51]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('Numerical 1', 
         Pipeline([
             ('log1p', FunctionTransformer(np.log1p, validate=True)),
             ('robust scaler', RobustScaler())
         ]), 
         ['CLTV', 'tenure']),
        
        ('Numerical 2', 
         Pipeline([
             ('robust scaler', RobustScaler())
         ]), 
         ['MonthlyCharges', 'TotalCharges']),
        
        ('Categorical One Hot', 
         Pipeline([
             ('One Hot Encoder', OneHotEncoder(drop='first'))
         ]), 
         ['gender', 'AgeGroup', 'Partner', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod', 'Contract']),

    ], 
    remainder='passthrough'
)



## Modeling & Evaluation

In [52]:
logreg = LogisticRegression()
knn = KNeighborsClassifier()
dt = DecisionTreeClassifier()
rf = RandomForestClassifier()
xgb = XGBClassifier()
lgbm = lgb.LGBMClassifier()

### Model Benchmarking : K-Fold

In [53]:
models = [logreg,knn,dt,rf,xgb,lgbm]
score=[]
rata=[]
std=[]

for i in models:
    skfold=StratifiedKFold(n_splits=5)
    estimator=Pipeline([
        ('preprocess',preprocessor),
        ('model',i)])
    model_cv=cross_val_score(estimator,x_train,y_train,cv=skfold,scoring='roc_auc')
    score.append(model_cv)
    rata.append(model_cv.mean())
    std.append(model_cv.std())
    
pd.DataFrame({'model':['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest', 'XGBoost', 'LightGBM'],'mean roc_auc':rata,'sdev':std}).set_index('model').sort_values(by='mean roc_auc',ascending=False)

[LightGBM] [Info] Number of positive: 957, number of negative: 2643
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000054 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 891
[LightGBM] [Info] Number of data points in the train set: 3600, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265833 -> initscore=-1.015867
[LightGBM] [Info] Start training from score -1.015867
[LightGBM] [Info] Number of positive: 957, number of negative: 2643
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000058 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 891
[LightGBM] [Info] Number of data points in the train set: 3600, number of used features: 31
[LightGBM] [Info] [binary:Bo

,mean roc_auc,sdev
model,,
Logistic Regression,0.854183,0.018055
LightGBM,0.833936,0.016516
Random Forest,0.832725,0.016122
XGBoost,0.823121,0.016130
KNN,0.780394,0.017795
Decision Tree,0.654360,0.017701


### Model Benchmarking : Test Data

In [54]:
models = [logreg,knn,dt,rf,xgb,lgbm]
score_roc_auc = []

def y_pred_func(i):
    estimator=Pipeline([
        ('preprocess',preprocessor),
        ('model',i)])
    x_train,x_test
    
    estimator.fit(x_train,y_train)
    return(estimator,estimator.predict(x_test),x_test)

for i,j in zip(models, ['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest', 'XGBoost','LightGBM']):
    estimator,y_pred,x_test = y_pred_func(i)
    y_predict_proba = estimator.predict_proba(x_test)[:,1]
    score_roc_auc.append(roc_auc_score(y_test,y_predict_proba))
    print(j,'\n', classification_report(y_test,y_pred))
    
pd.DataFrame({'model':['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest', 'XGBoost','LightGBM'],
             'roc_auc score':score_roc_auc}).set_index('model').sort_values(by='roc_auc score',ascending=False)

Logistic Regression 
               precision    recall  f1-score   support

       False       0.83      0.91      0.87       826
        True       0.67      0.50      0.57       299

    accuracy                           0.80      1125
   macro avg       0.75      0.70      0.72      1125
weighted avg       0.79      0.80      0.79      1125

KNN 
               precision    recall  f1-score   support

       False       0.83      0.85      0.84       826
        True       0.57      0.53      0.55       299

    accuracy                           0.77      1125
   macro avg       0.70      0.69      0.70      1125
weighted avg       0.76      0.77      0.77      1125

Decision Tree 
               precision    recall  f1-score   support

       False       0.82      0.83      0.82       826
        True       0.51      0.49      0.50       299

    accuracy                           0.74      1125
   macro avg       0.66      0.66      0.66      1125
weighted avg       0.74      0

,roc_auc score
model,
Logistic Regression,0.838663
LightGBM,0.826806
Random Forest,0.819483
XGBoost,0.811579
KNN,0.777335
Decision Tree,0.660325
